## Student setup

Run the modules in order: M1 creates the handoff dataset consumed by M2, M3, and M4. Keep the master dataset in approved private storage; it is intentionally excluded from this repository. Set `MEDML_MASTER_DATASET_PATH` for Module 1 and `MEDML_OUTPUT_DIR` if outputs should persist outside the repository.


# M3 | Critical outcome analysis

This notebook asks whether information available at ED triage can identify a stay with **critical outcome**. It adapts the uploaded/source analysis (Critical outcome categorized Analysis NEW.ipynb, Critical_outcome_Statististics.ipynb, Task_2_Model_critical_triage.ipynb) to the extracted teaching CSV. The target definition is explicit, the predictor boundary is enforced, and all performance results are recomputed locally.

## Clinical question and learning objectives

**Question:** Can a triage-time model rank ED stays by the likelihood of critical outcome?

Students will:

- reproduce and audit the target definition;
- separate target construction from predictors;
- quantify prevalence and the consequences of class imbalance;
- compare a dummy, logistic regression, random forest, and gradient boosting baseline;
- interpret confusion matrices, sensitivity, specificity, PPV, NPV, F1, ROC AUC, and average precision;
- inspect threshold, error, subgroup, calibration, and feature-influence behavior.

This is a retrospective benchmark exercise. A score is not a diagnosis or a deployment recommendation.

## Target definition: what counts as critical outcome?

`outcome_critical` is true when either `outcome_inhospital_mortality` or `outcome_icu_transfer_12h` is true. This composite emphasizes severe early outcomes but combines two clinically different events; students should inspect both components before treating it as one phenomenon.

The target is a retrospective ground-truth-like label from the source table. It is not information that would be available at the moment of triage. We therefore verify it first and exclude every field used to construct it, plus downstream fields, from the feature matrix.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
import os

repo_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (
        candidate
        for candidate in repo_candidates
        if (candidate / "notebooks").is_dir() and (candidate / "data").is_dir()
    ),
    Path.cwd(),
)
output_override = os.getenv("MEDML_OUTPUT_DIR")
OUTPUT_DIR = Path(output_override).expanduser() if output_override else REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = OUTPUT_DIR / "M1_dataset_for_next_module.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "M1 handoff not found. Run Module 1 first, or set MEDML_OUTPUT_DIR "
        "to the folder containing M1_dataset_for_next_module.csv."
    )
ROOT_DIR = REPO_ROOT
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded M1-ready dataset: {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
print(f"Critical outcomes: {int(df['outcome_critical'].sum()):,} ({df['outcome_critical'].mean() * 100:.2f}%)")

In [ ]:
TARGET_COLUMN = "outcome_critical"
MODULE_PREFIX = "M3"
critical_from_components = df["outcome_inhospital_mortality"] | df["outcome_icu_transfer_12h"]
assert (critical_from_components == df[TARGET_COLUMN]).all()
component_summary = pd.DataFrame({
    "event": ["in-hospital mortality", "ICU transfer within 12 hours", "critical composite"],
    "count": [int(df["outcome_inhospital_mortality"].sum()), int(df["outcome_icu_transfer_12h"].sum()), int(df[TARGET_COLUMN].sum())],
    "prevalence_pct": [df["outcome_inhospital_mortality"].mean() * 100, df["outcome_icu_transfer_12h"].mean() * 100, df[TARGET_COLUMN].mean() * 100],
})
display(component_summary.round(2))

## 1. Describe the target before modeling

Prevalence is the fraction of positive stays. It affects the prior probability of a positive prediction and therefore affects PPV and NPV. Report both counts and percentages, and inspect outcome prevalence across a small number of clinically legible groups before fitting anything.

In [ ]:
critical_by_acuity = (
    df.groupby("triage_acuity", dropna=False)
    .agg(stays=("stay_id", "size"), critical_count=(TARGET_COLUMN, "sum"), critical_pct=(TARGET_COLUMN, "mean"))
    .reset_index()
)
critical_by_acuity["critical_pct"] *= 100
display(critical_by_acuity.round(2))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=critical_by_acuity, x="triage_acuity", y="critical_pct", color="#0d9488", ax=axes[0])
axes[0].set(title="Critical outcome by triage acuity", xlabel="Triage acuity", ylabel="Critical outcome (%)")
sns.countplot(data=df, x=TARGET_COLUMN, color="#ea580c", ax=axes[1])
axes[1].set(title="Class counts", xlabel="Critical outcome", ylabel="Stays")
plt.tight_layout()
plt.show()

## 2. Declare the triage-time predictor boundary

We use the benchmark feature family: demographics/context, prior ED/hospital/ICU utilization, triage measurements, chief-complaint flags, and CCI/ECI comorbidity indicators. Raw `chiefcomplaint` text is intentionally outside this tabular baseline.

Exclude identifiers, timestamps, disposition, ED departure information, ED LOS, hospitalization or critical labels, mortality/ICU timing, revisits, later ED measurements, and medication counts. A clinically plausible field is still leakage if it is recorded after the decision point.

In [ ]:
triage_feature_candidates = [
    ["age"],
    ["gender"],
    ["race"],
    ["arrival_transport"],
    ["triage_temperature_celsius_iterative", "triage_temperature_celsius", "triage_temperature"],
    ["triage_heartrate_iterative", "triage_heartrate"],
    ["triage_resprate_iterative", "triage_resprate"],
    ["triage_o2sat_iterative", "triage_o2sat"],
    ["triage_sbp_iterative", "triage_sbp"],
    ["triage_dbp_iterative", "triage_dbp"],
    ["triage_pain_iterative", "triage_pain"],
    ["triage_acuity_iterative", "triage_acuity"],
]
triage_features = [
    next((candidate for candidate in candidates if candidate in df.columns), None)
    for candidates in triage_feature_candidates
]
triage_features = [column for column in triage_features if column is not None]
prior_features = [
    "n_ed_30d", "n_ed_90d", "n_ed_365d", "n_hosp_30d", "n_hosp_90d",
    "n_hosp_365d", "n_icu_30d", "n_icu_90d", "n_icu_365d",
]
complaint_features = [column for column in df.columns if column.startswith("chiefcom_")]
comorbidity_features = [
    column for column in df.columns
    if column.startswith("cci_") or column.startswith("eci_")
]
FEATURES = [
    column for column in triage_features + prior_features + complaint_features + comorbidity_features
    if column in df.columns
]
CATEGORICAL = [column for column in ["gender", "race", "arrival_transport"] if column in FEATURES]
NUMERIC = [column for column in FEATURES if column not in CATEGORICAL]
print(f"Predictors in the declared triage-time boundary: {len(FEATURES)}")
print("Triage predictors:", triage_features)
print("Categorical:", CATEGORICAL)
print("Numeric/binary:", len(NUMERIC))

In [ ]:
excluded_columns = [
    column for column in df.columns
    if column in {
        "index", "subject_id", "hadm_id", "stay_id", "intime", "outtime", "admittime",
        "dischtime", "deathtime", "edregtime", "edouttime", "disposition", "ed_los",
        "ed_los_hours", "outcome_inhospital_mortality", "outcome_icu_transfer_12h",
        "time_to_icu_transfer", "time_to_icu_transfer_hours", "outcome_hospitalization",
        "outcome_critical", "next_ed_visit_time", "next_ed_visit_time_diff",
        "next_ed_visit_time_diff_days", "outcome_ed_revisit_3d", "ed_temperature_last",
        "ed_heartrate_last", "ed_resprate_last", "ed_o2sat_last", "ed_sbp_last",
        "ed_dbp_last", "ed_pain_last", "n_med", "n_medrecon",
    }
]
print("Excluded fields represented in the source:")
print(excluded_columns)
assert not set(FEATURES) & set(excluded_columns)

## 3. Patient-disjoint split and training-only preprocessing

We use a reproducible 30,000-row teaching sample. A patient may have several ED stays, so `subject_id` is the grouping key. The test partition is held out while models are developed. Imputation, missingness indicators, scaling, and one-hot encoding live inside each pipeline and are fitted on training rows only.

In [ ]:
model_df = df.sample(n=min(50000, len(df)), random_state=42).reset_index(drop=True)
X = model_df[FEATURES]
y = model_df[TARGET_COLUMN].astype(int)
groups = model_df["subject_id"]
from sklearn.model_selection import GroupShuffleSplit
splitter = GroupShuffleSplit(n_splits=1, test_size=.2, random_state=42)
train_index, test_index = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_index], X.iloc[test_index]
y_train, y_test = y.iloc[train_index], y.iloc[test_index]
groups_train, groups_test = groups.iloc[train_index], groups.iloc[test_index]
print(f"Train rows: {len(X_train):,}; test rows: {len(X_test):,}")
print(f"Train patients: {groups_train.nunique():,}; test patients: {groups_test.nunique():,}")
print("Patient overlap:", len(set(groups_train) & set(groups_test)))
print(f"Target prevalence: train={y_train.mean():.3f}; test={y_test.mean():.3f}")
assert set(groups_train).isdisjoint(set(groups_test))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def make_preprocessor():
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                    ("scaler", StandardScaler()),
                ]),
                NUMERIC,
            ),
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", encoder),
                ]),
                CATEGORICAL,
            ),
        ],
        remainder="drop",
    )


def make_pipeline(estimator):
    return Pipeline([("preprocessor", make_preprocessor()), ("model", estimator)])

## 4. Class imbalance and training-only sampling

Critical outcomes are uncommon in this cohort. Sampling can change the class composition seen during training, but it must be fitted on the training partition only. The held-out test partition stays at its original prevalence so evaluation represents the population the model is meant to encounter.

We compare no resampling, random under-sampling, random over-sampling, SMOTE, and SMOTE-Tomek. Synthetic methods operate after the shared preprocessing pipeline has converted missing values and categories into numeric features; their generated rows are training artefacts, not new patients.

In [ ]:
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

sampling_methods = {
    "original training": None,
    "random under-sampling": RandomUnderSampler(random_state=42),
    "random over-sampling": RandomOverSampler(random_state=42),
    "SMOTE": SMOTE(random_state=42, k_neighbors=5),
    "SMOTE-Tomek": SMOTETomek(random_state=42),
}

sampling_preprocessor = make_preprocessor()
X_train_encoded = sampling_preprocessor.fit_transform(X_train, y_train)
sampled_training_sets = {}
sampling_rows = []
for method_name, sampler in sampling_methods.items():
    if sampler is None:
        sampled_X, sampled_y = X_train_encoded, y_train.to_numpy()
    else:
        sampled_X, sampled_y = sampler.fit_resample(X_train_encoded, y_train)
    sampled_training_sets[method_name] = (sampled_X, sampled_y)
    sampling_rows.extend([
        {"sampling_method": method_name, "class": 0, "rows": int((sampled_y == 0).sum()), "prevalence_pct": float((sampled_y == 0).mean() * 100)},
        {"sampling_method": method_name, "class": 1, "rows": int((sampled_y == 1).sum()), "prevalence_pct": float((sampled_y == 1).mean() * 100)},
    ])

sampling_summary = pd.DataFrame(sampling_rows)
display(sampling_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=sampling_summary, x="sampling_method", y="rows", hue="class", palette=["#94a3b8", "#ea580c"], ax=axes[0])
axes[0].set(title="Training rows after each sampling strategy", xlabel="", ylabel="Rows")
axes[0].tick_params(axis="x", rotation=25)
axes[0].legend(title="Critical outcome", labels=["0 = no", "1 = yes"])
sns.barplot(data=sampling_summary, x="sampling_method", y="prevalence_pct", hue="class", palette=["#94a3b8", "#ea580c"], ax=axes[1])
axes[1].set(title="Class composition after sampling", xlabel="", ylabel="Class share (%)")
axes[1].tick_params(axis="x", rotation=25)
axes[1].axhline(y_train.mean() * 100, linestyle=":", color="black", label=f"original positive prevalence={y_train.mean():.1%}")
axes[1].legend(title="Critical outcome")
plt.tight_layout()
plt.show()

print(f"Untouched test prevalence: {y_test.mean():.2%}")
assert sampling_summary.loc[sampling_summary["sampling_method"] == "original training", "rows"].sum() == len(y_train)
assert y_test.mean() == model_df.loc[test_index, TARGET_COLUMN].mean()

## 4. Candidate models and evaluation metrics

The prior dummy is the minimum baseline. Logistic regression is a transparent linear reference. Random forest can represent nonlinear interactions, while gradient boosting builds a sequence of small trees. The comparison is fair only when all models use the same rows, target, feature list, and held-out test partition.

At a threshold of 0.5, sensitivity is the fraction of positive outcomes found; specificity is the fraction of negatives left negative; PPV is the fraction of alerts that are positive; NPV is the fraction of negative predictions that are negative. ROC AUC and average precision summarize ranking across thresholds.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


def classification_metrics(y_true, probabilities, threshold=0.5):
    predictions = np.asarray(probabilities) >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    has_both_classes = len(np.unique(y_true)) == 2
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, predictions),
        "sensitivity": recall_score(y_true, predictions, zero_division=0),
        "specificity": tn / (tn + fp) if tn + fp else np.nan,
        "ppv": precision_score(y_true, predictions, zero_division=0),
        "npv": tn / (tn + fn) if tn + fn else np.nan,
        "f1": f1_score(y_true, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probabilities) if has_both_classes else np.nan,
        "average_precision": average_precision_score(y_true, probabilities) if has_both_classes else np.nan,
        "true_positive": tp,
        "false_positive": fp,
        "true_negative": tn,
        "false_negative": fn,
    }

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

models = {
    "prior dummy": DummyClassifier(strategy="prior"),
    "logistic regression": make_pipeline(LogisticRegression(max_iter=500, class_weight="balanced", random_state=42)),
    "decision tree": make_pipeline(DecisionTreeClassifier(max_depth=8, min_samples_leaf=25, class_weight="balanced", random_state=42)),
    "random forest": make_pipeline(RandomForestClassifier(n_estimators=80, min_samples_leaf=3, class_weight="balanced_subsample", n_jobs=1, random_state=42)),
    "gradient boosting": make_pipeline(GradientBoostingClassifier(n_estimators=60, max_depth=2, learning_rate=.08, random_state=42)),
}
fitted_models = {}
probabilities = {}
result_rows = []
for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    fitted_models[name] = estimator
    probabilities[name] = estimator.predict_proba(X_test)[:, 1]
    result_rows.append({"model": name, **classification_metrics(y_test, probabilities[name])})
model_results = pd.DataFrame(result_rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
display(model_results.round(3))

## 5. Compare models across sampling strategies

Resampling is another modeling choice, so it must be evaluated rather than assumed to help. Each classifier below is trained on one training-only sampling strategy and evaluated on the same untouched test partition. This makes the comparison about the training distribution, not a changing evaluation population.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_recall_curve, roc_curve

X_test_encoded = sampling_preprocessor.transform(X_test)
sampling_model_factories = {
    "logistic regression": lambda: LogisticRegression(max_iter=500, random_state=42),
    "decision tree": lambda: DecisionTreeClassifier(max_depth=8, min_samples_leaf=25, random_state=42),
    "random forest": lambda: RandomForestClassifier(
        n_estimators=80, min_samples_leaf=3, n_jobs=1, random_state=42
    ),
    "gradient boosting": lambda: GradientBoostingClassifier(
        n_estimators=60, max_depth=2, learning_rate=.08, random_state=42
    ),
}
sampling_model_results_rows = []
sampling_probabilities = {}
for sampling_method, (sampled_X, sampled_y) in sampled_training_sets.items():
    sampling_probabilities[sampling_method] = {}
    for model_name, factory in sampling_model_factories.items():
        estimator = factory()
        estimator.fit(sampled_X, sampled_y)
        model_probabilities = estimator.predict_proba(X_test_encoded)[:, 1]
        sampling_probabilities[sampling_method][model_name] = model_probabilities
        sampling_model_results_rows.append({
            "sampling_method": sampling_method,
            "model": model_name,
            "roc_auc": roc_auc_score(y_test, model_probabilities),
            "average_precision": average_precision_score(y_test, model_probabilities),
            **{
                key: value
                for key, value in classification_metrics(y_test, model_probabilities).items()
                if key in {"sensitivity", "specificity", "ppv", "npv", "f1"}
            },
        })

sampling_model_results = pd.DataFrame(sampling_model_results_rows).sort_values(
    ["roc_auc", "average_precision"], ascending=False
).reset_index(drop=True)
display(sampling_model_results.round(3))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
auc_table = sampling_model_results.pivot(index="model", columns="sampling_method", values="roc_auc")
ap_table = sampling_model_results.pivot(index="model", columns="sampling_method", values="average_precision")
sns.heatmap(auc_table, annot=True, fmt=".3f", cmap="YlGnBu", vmin=.45, vmax=1, ax=axes[0])
axes[0].set(title="AUROC by classifier and sampling method", xlabel="", ylabel="")
sns.heatmap(ap_table, annot=True, fmt=".3f", cmap="Oranges", vmin=0, vmax=1, ax=axes[1])
axes[1].set(title="Average precision by classifier and sampling method", xlabel="", ylabel="")
plt.tight_layout()
plt.show()

sampling_roc_model = "logistic regression"
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for sampling_method, model_probabilities in sampling_probabilities.items():
    method_probabilities = model_probabilities[sampling_roc_model]
    fpr, tpr, _ = roc_curve(y_test, method_probabilities)
    precision, recall, _ = precision_recall_curve(y_test, method_probabilities)
    axes[0].plot(fpr, tpr, label=f"{sampling_method}: {roc_auc_score(y_test, method_probabilities):.3f}")
    axes[1].plot(recall, precision, label=f"{sampling_method}: AP={average_precision_score(y_test, method_probabilities):.3f}")
axes[0].plot([0, 1], [0, 1], ":", color="black")
axes[1].axhline(y_test.mean(), linestyle=":", color="black", label=f"test prevalence={y_test.mean():.3f}")
axes[0].set(title="Logistic ROC by sampling strategy", xlabel="False-positive rate", ylabel="Sensitivity")
axes[1].set(title="Logistic precision-recall by sampling strategy", xlabel="Recall / sensitivity", ylabel="Precision / PPV")
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. ROC and precision-recall views

ROC AUC is useful for ranking but can look reassuring when the number of false alerts is operationally large. Precision-recall curves make the positive class and its prevalence visible. Read the curves together with the confusion matrix at the threshold a real user would actually apply.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, model_probabilities in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, model_probabilities)
    precision, recall, _ = precision_recall_curve(y_test, model_probabilities)
    axes[0].plot(fpr, tpr, label=f"{name}: {roc_auc_score(y_test, model_probabilities):.3f}")
    axes[1].plot(recall, precision, label=f"{name}: AP={average_precision_score(y_test, model_probabilities):.3f}")
axes[0].plot([0, 1], [0, 1], ":", color="black")
axes[1].axhline(y_test.mean(), linestyle=":", color="black", label=f"prevalence={y_test.mean():.3f}")
axes[0].set(title="ROC comparison", xlabel="False-positive rate", ylabel="Sensitivity")
axes[1].set(title="Precision-recall comparison", xlabel="Sensitivity / precision", ylabel="Precision / PPV")
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Threshold analysis and confusion matrix

A threshold is a policy choice. Lowering it can capture more positive outcomes while increasing false positives; raising it can reduce alerts while missing more positives. Use the table to connect a model score to a named action. Do not call 0.5 “the clinical threshold” without a decision specification.

In [ ]:
selected_model_name = "logistic regression"
selected_probabilities = probabilities[selected_model_name]
threshold_results = pd.DataFrame([
    {"model": selected_model_name, **classification_metrics(y_test, selected_probabilities, threshold)}
    for threshold in [.10, .25, .40, .50, .60, .75]
])
display(threshold_results.round(3))
selected_threshold = .50
selected_predictions = selected_probabilities >= selected_threshold
confusion = pd.DataFrame(
    confusion_matrix(y_test, selected_predictions, labels=[0, 1]),
    index=["observed 0", "observed 1"],
    columns=["predicted 0", "predicted 1"],
)
display(confusion)

## 7. Calibration: does a probability mean what it says?

Discrimination asks whether higher scores tend to belong to positive cases. Calibration asks whether groups assigned a probability of, for example, 0.70 experience the outcome about 70% of the time. A model can rank cases well and still be poorly calibrated. The Brier score is the mean squared error of the predicted probabilities; lower is better on the same test population.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

fraction_positive, mean_predicted = calibration_curve(
    y_test, selected_probabilities, n_bins=10, strategy="quantile"
)
calibration_results = pd.DataFrame({
    "mean_predicted_probability": mean_predicted,
    "observed_fraction_positive": fraction_positive,
})
print(f"Brier score: {brier_score_loss(y_test, selected_probabilities):.4f}")
display(calibration_results.round(3))
plt.figure(figsize=(6, 6))
plt.plot(mean_predicted, fraction_positive, "o-", color="#ea580c", label="model")
plt.plot([0, 1], [0, 1], ":", color="black", label="perfect calibration")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed fraction positive")
plt.title(f"Calibration for {selected_model_name}")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Cross-validation inside the training partition

    The test partition is not used for model selection. Grouped three-fold cross-validation inside training gives a sense of variation when patients are held apart in each fold. It is not an external validation study and cannot repair target bias, measurement bias, or a non-transportable workflow.

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_score
cv = GroupKFold(n_splits=3)
cv_pipeline = make_pipeline(LogisticRegression(max_iter=500, class_weight="balanced", random_state=42))
cv_scores = cross_val_score(cv_pipeline, X_train, y_train, groups=groups_train, cv=cv, scoring="roc_auc", n_jobs=1)
cv_summary = pd.DataFrame({"fold": np.arange(1, len(cv_scores) + 1), "roc_auc": cv_scores})
display(cv_summary.round(3))
print(f"Grouped CV ROC AUC mean={cv_scores.mean():.3f}; SD={cv_scores.std(ddof=1):.3f}")

## 9. Global feature influence: useful but bounded

Permutation importance shuffles one raw input in held-out rows and measures the resulting performance change through the complete pipeline. It answers “how much does this input help this model on these rows?” It does not answer “what causes the outcome?” Correlated variables can share or hide importance, and workflow variables may not transport.

In [ ]:
from sklearn.inspection import permutation_importance
importance_sample = X_test.sample(n=min(5000, len(X_test)), random_state=42)
importance_labels = y_test.loc[importance_sample.index]
importance = permutation_importance(
    fitted_models[selected_model_name], importance_sample, importance_labels,
    scoring="roc_auc", n_repeats=3, random_state=42, n_jobs=1,
)
importance_table = (
    pd.DataFrame({"feature": importance_sample.columns, "mean_auc_drop": importance.importances_mean, "sd_auc_drop": importance.importances_std})
    .sort_values("mean_auc_drop", ascending=False)
)
display(importance_table.head(20).round(4))

## 10. Subgroup performance and limitations

Overall performance can conceal different behavior across gender, race, or arrival transport. Report subgroup size and prevalence next to sensitivity, specificity, PPV, NPV, and AUC. Small groups and intersectional groups need uncertainty intervals in a full study; a descriptive table is not a complete fairness analysis.

In [ ]:
test_view = X_test[[column for column in ["gender", "race", "arrival_transport"] if column in X_test.columns]].copy()
test_view["observed"] = y_test.to_numpy()
test_view["probability"] = selected_probabilities
subgroup_rows = []
for group_column in ["gender", "race", "arrival_transport"]:
    for group_value, group in test_view.groupby(group_column, dropna=False):
        if len(group) < 50:
            continue
        subgroup_rows.append({
            "group_variable": group_column,
            "group": str(group_value),
            "n": len(group),
            "prevalence": group["observed"].mean(),
            **{key: value for key, value in classification_metrics(group["observed"], group["probability"]).items() if key in {"sensitivity", "specificity", "ppv", "npv", "roc_auc"}},
        })
subgroup_results = pd.DataFrame(subgroup_rows)
display(subgroup_results.round(3))

### Student accountability lab

Choose one model and one operating threshold. Write a short model card containing the intended population, index time, target definition, predictor exclusions, held-out metrics, one error concern, one subgroup concern, and the evidence still required before clinical use. Separate observed evidence from hypotheses and from missing evidence.

In [ ]:
predictions_table = X_test[[column for column in ['triage_acuity_iterative',"gender", "race", "arrival_transport"] if column in X_test.columns]].copy()
predictions_table["observed_target"] = y_test.to_numpy()
predictions_table["predicted_probability"] = selected_probabilities
predictions_table["predicted_class"] = selected_predictions.astype(int)
predictions_table["error_type"] = np.select(
    [predictions_table["observed_target"].eq(1) & predictions_table["predicted_class"].eq(0), predictions_table["observed_target"].eq(0) & predictions_table["predicted_class"].eq(1)],
    ["false negative", "false positive"], default="correct",
)
error_examples = predictions_table.loc[predictions_table["error_type"] != "correct"].sort_values("predicted_probability", ascending=False)
display(error_examples.head(20))

## M3 handoff and final questions

The notebook exports computed results, not copied historical claims. Ask:

1. What information is available at triage and what is downstream?
2. Which error matters most for the intended action, and why?
3. How does prevalence change PPV and NPV?
4. What does feature importance fail to establish?
5. What validation, calibration, fairness, and workflow evidence is missing?

In [ ]:
M3_or_M4_tables = {
    "model_results.csv": model_results,
    "sampling_summary.csv": sampling_summary,
    "sampling_model_results.csv": sampling_model_results,
    "threshold_results.csv": threshold_results,
    "calibration_results.csv": calibration_results,
    "cv_results.csv": cv_summary,
    "feature_importance.csv": importance_table,
    "subgroup_results.csv": subgroup_results,
    "error_examples.csv": error_examples,
}
for filename, table in M3_or_M4_tables.items():
    table.to_csv(OUTPUT_DIR / f"{MODULE_PREFIX}_{filename}", index=False)
print(f"Wrote {len(M3_or_M4_tables)} {MODULE_PREFIX} tables to {OUTPUT_DIR}")